In [ ]:
# Paths (added for the public repo). Paths below are relative to this notebook;
# override with the DATA_ROOT / RUNS_ROOT environment variables.
import os
DATA_ROOT = os.environ.get('DATA_ROOT', '../../data')   # data/blouse, data/aihub_official
RUNS_ROOT = os.environ.get('RUNS_ROOT', '../../runs')   # checkpoints / metrics are written here


# 폐의류 재활용 분류 — **B4: Grad-CAM + bbox IoU (설명가능 비전 AI)**

Ablation Study 의 **B4** 단계입니다. B0/B1/B2 에서 학습한 체크포인트(`best.pt`)를
그대로 불러와 **Grad-CAM** 으로 모델이 어디를 보고 상태를 판단했는지 시각화하고,
AI-Hub 라벨의 `pollution`(오염)·`damage`(손상) **bbox 를 GT 로** 삼아
**WSOL(약지도 위치추정) 정량 지표**를 계산합니다.

**산출 지표 (제안서 §3 XAI 평가 포인트)**
- **GT-Known IoU**: Grad-CAM 으로 만든 예측 박스 ↔ 오염/손상 GT 박스의 IoU (평균)
- **Loc@0.5**: IoU ≥ 0.5 비율 (정상적으로 옳은 영역을 짚은 비율)
- **Pointing-Game**: Grad-CAM 최고점이 GT 박스 안에 들어간 비율 (가장 강건한 지표)

> ⚙️ 한 가지만 정하면 됩니다 — 셀 `Config` 의 `MODEL_VARIANT` 를 `'B0'|'B1'|'B2'` 중
> 평가할 모델로 설정하세요. 세 모델 모두 동일 코드로 평가되어 **공정 비교**가 됩니다.

**중요**: 이 평가는 **학습/추론과 동일한 전처리(`eval_tf`, 224×224 직접 리사이즈)**·
**동일 test split(seed=42)** 를 그대로 재현합니다. 모델이 본 적 없는 test 이미지에서만 평가합니다.


## 0. 환경 설치

In [ ]:
# Grad-CAM 은 직접 구현(외부 라이브러리 불필요). scipy(연결요소)·seaborn 만 확인 설치
!pip -q install scipy seaborn scikit-learn

## 1. Imports & 재현성 시드

In [ ]:
import os, json, random, time, math
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy import ndimage
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights, densenet121, DenseNet121_Weights

from sklearn.model_selection import train_test_split

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Torch:', torch.__version__, '| Device:', DEVICE)

def seed_everything(seed: int = 42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(42)

# === [TRUNCATED IMAGE FIX] 잘린(손상) 이미지도 읽도록 허용 ===
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False


## 2. 설정 (Config)

`MODEL_VARIANT` 만 바꾸면 B0/B1/B2 를 동일 코드로 평가합니다.
`bbox_format` 은 AI-Hub bbox 표기에 맞추세요 (기본 `xywh` = [x, y, w, h];
만약 [x1,y1,x2,y2] 형식이면 `'xyxy'` 로). 셀 4에서 자동 점검 메시지를 출력합니다.

In [ ]:
@dataclass
class Config:
    # ── 평가할 모델 선택 ───────────────────────────────────────────
    MODEL_VARIANT: str = 'B2MS_IMP'          # 'B0' | 'B1' | 'B2MS' | 'IMPROVED'

    # 데이터 (B0~B2 와 동일 경로여야 동일 split 재현)
    data_root: str = f'{DATA_ROOT}/blouse'

    # 각 변형의 체크포인트 위치 (학습 때 output_dir 와 일치시켜야 함)
    ckpt_B0: str = f'{RUNS_ROOT}/baseline_resnet50/best.pt'
    ckpt_B1: str = f'{RUNS_ROOT}/baseline_B1_texture/best.pt'
    ckpt_B2MS: str = f'{RUNS_ROOT}/baseline_B2_multiscale/best.pt'
    ckpt_IMPROVED: str = f'{RUNS_ROOT}/B_improved/best.pt'
    ckpt_B2MSIMP: str  = f'{RUNS_ROOT}/B2MS_improved/best.pt'

    output_dir: str = f'{RUNS_ROOT}/B4_xai'

    main_classes: tuple = ('재사용', '재활용', '폐기')
    img_size: int = 224
    seed: int = 42
    val_ratio: float = 0.15
    test_ratio: float = 0.15

    # B1 / B2 재구성용 (학습 때 값과 동일하게)
    texture_mid_ch: int = 32           # B1
    dropout_p: float = 0.3             # IMPROVED 재구성용 (학습 때 값과 동일)

    # ── XAI 평가 하이퍼파라미터 ────────────────────────────────────
    bbox_format: str = 'xywh'          # 'xywh' | 'xyxy'
    cam_threshold: float = 0.5         # CAM 박스화 임계 (max 의 비율)
    cam_class: str = 'gt'              # 'gt'(GT-Known) | 'pred'(Top-1 Loc)
    max_eval: int = 600                # 전체 평가 표본 상한(속도). None 이면 전부
    n_viz: int = 12                    # 정성 시각화 장수

cfg = Config()
ckpt_map = {'B0': cfg.ckpt_B0, 'B1': cfg.ckpt_B1,
            'B2MS': cfg.ckpt_B2MS, 'IMPROVED': cfg.ckpt_IMPROVED,
            'B2MS_IMP': cfg.ckpt_B2MSIMP}
CKPT = ckpt_map[cfg.MODEL_VARIANT]
OUT  = Path(cfg.output_dir) / cfg.MODEL_VARIANT
OUT.mkdir(parents=True, exist_ok=True)
print('Variant :', cfg.MODEL_VARIANT)
print('Ckpt    :', CKPT)
print('Out dir :', OUT)

## 3. 데이터 — 매니페스트 빌드 (B0~B2 와 동일 로직)

In [ ]:
# (로컬 실행) Google Drive 마운트 불필요 — 실행해도 아무 동작 안 합니다.
print('local run: drive mount skipped')


In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
STATE_MAP = {'dispose': '폐기', 'recycle': '재활용', 'reusable': '재사용'}
PREFIX_MAP = {'DP': '폐기', 'RC': '재활용', 'RU': '재사용'}

def state_from_filename(name: str):
    pref = Path(name).stem.split('_')[0].upper()
    return PREFIX_MAP.get(pref)

def parse_label_json(json_path: Path, image_lookup: dict):
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception:
        return None
    meta = data.get('meta_information', {})
    big  = meta.get('big_category', '').strip().lower()
    med  = meta.get('medium_category', 'unknown').strip().lower()
    fname = meta.get('file_name') or f'{json_path.stem}.jpg'
    main = STATE_MAP.get(big) or state_from_filename(fname)
    if main is None:
        return None
    img_path = image_lookup.get(fname) or image_lookup.get(Path(fname).stem)
    if img_path is None:
        return None
    ann = data.get('annotations', {})
    return {
        'file_path': str(img_path),
        'main': main,
        'sub':  med,
        'clothes_bbox':    [b['bbox'] for b in ann.get('clothes', [])],
        'pollution_bbox':  [b['bbox'] for b in ann.get('pollution', [])],
        'damage_bbox':     [b['bbox'] for b in ann.get('damage', [])],
        'attachment_bbox': [b['bbox'] for b in ann.get('attachment', [])],
    }

def build_manifest(data_root: str) -> pd.DataFrame:
    data_root = Path(data_root)
    if not data_root.exists():
        raise FileNotFoundError(f'data_root not found: {data_root}')
    print('Indexing images...', flush=True)
    image_lookup = {}
    for p in data_root.rglob('*'):
        if p.suffix.lower() in IMG_EXTS:
            image_lookup[p.name] = p
            image_lookup.setdefault(p.stem, p)
    print(f'  {len(image_lookup)} image files indexed')
    print('Parsing JSON labels...', flush=True)
    rows, skipped = [], 0
    for jp in data_root.rglob('*.json'):
        r = parse_label_json(jp, image_lookup)
        if r is None:
            skipped += 1; continue
        rows.append(r)
    df = pd.DataFrame(rows).drop_duplicates(subset='file_path').reset_index(drop=True)
    print(f'  parsed {len(df)} samples, skipped {skipped}')
    return df

df = build_manifest(cfg.data_root)
assert len(df) > 0, '매니페스트가 비었습니다. cfg.data_root 확인.'
print('\n오염 bbox 보유:', (df['pollution_bbox'].apply(len) > 0).sum(),
      '| 손상 bbox 보유:', (df['damage_bbox'].apply(len) > 0).sum())

## 4. Test split 재현 (Stratified, seed=42)

B0~B2 와 **완전히 같은 코드/seed** 로 분할하므로 모델이 학습한 그 test set 을 그대로 평가합니다.
(만약 B0 단계에서 split 파일 리스트를 csv 로 저장해 두었다면, 아래 대신 그 csv 를 읽으세요.)

In [ ]:
df = df[df['main'].isin(cfg.main_classes)].reset_index(drop=True)
main2idx = {c: i for i, c in enumerate(cfg.main_classes)}
df['main_idx'] = df['main'].map(main2idx)

SUB_CLASSES = tuple(df['sub'].value_counts().index.tolist())
sub2idx = {c: i for i, c in enumerate(SUB_CLASSES)}
df['sub_idx'] = df['sub'].map(sub2idx)

import os as _os
_split_dir = f'{RUNS_ROOT}/_split'
_tef = _os.path.join(_split_dir,'test_files.csv')
if _os.path.exists(_tef):
    _te = set(pd.read_csv(_tef)['file_path'])
    test_df = df[df['file_path'].isin(_te)].reset_index(drop=True)
    print(f'[split: B0 CSV 재사용] Test {len(test_df)} | sub_classes={len(SUB_CLASSES)}')
else:
    print('[split: B0 CSV 없음 -> 자체 분할. B0 먼저 실행 권장]')
    train_df, temp_df = train_test_split(df, test_size=cfg.val_ratio + cfg.test_ratio,
        stratify=df['main_idx'], random_state=cfg.seed)
    _vr = cfg.val_ratio / (cfg.val_ratio + cfg.test_ratio)
    val_df, test_df = train_test_split(temp_df, test_size=1 - _vr,
        stratify=temp_df['main_idx'], random_state=cfg.seed)
    test_df = test_df.reset_index(drop=True)
    print(f'Test {len(test_df)} | sub_classes={len(SUB_CLASSES)}')

def _flatten_boxes(x):
    # 숫자 4개짜리 박스만 재귀적으로 추출 (껍데기 몇 겹이든, np.array/NaN/빈값 처리)
    if x is None:
        return []
    if hasattr(x, 'tolist'):        # numpy array 등
        x = x.tolist()
    if isinstance(x, float):        # NaN 등
        return []
    if (isinstance(x, (list, tuple)) and len(x) == 4
            and all(isinstance(v, (int, float)) for v in x)):
        return [[float(v) for v in x]]
    if isinstance(x, (list, tuple)):
        out = []
        for e in x:
            out.extend(_flatten_boxes(e))
        return out
    return []

test_df['gt_boxes'] = test_df.apply(
    lambda r: _flatten_boxes(r['pollution_bbox']) + _flatten_boxes(r['damage_bbox']),
    axis=1
)
eval_df = test_df[test_df['gt_boxes'].apply(len) > 0].reset_index(drop=True)
print(f'GT 박스 보유 test 샘플(평가 대상): {len(eval_df)}')
print('클래스 분포:', dict(eval_df['main'].value_counts()))


## 5. 전처리 — 학습/추론과 동일한 eval_tf

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

eval_tf = transforms.Compose([
    transforms.Resize((cfg.img_size, cfg.img_size)),   # 종횡비 무시 직접 리사이즈 (학습과 동일)
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

_mean = torch.tensor(MEAN).view(3, 1, 1)
_std  = torch.tensor(STD).view(3, 1, 1)
def denormalize(t):
    '''정규화 텐서 -> [0,1] numpy(H,W,3), 시각화용'''
    img = (t.cpu() * _std + _mean).clamp(0, 1)
    return img.permute(1, 2, 0).numpy()

## 6. bbox 좌표 도구 — 원본 픽셀 → 224 공간

`eval_tf` 가 원본 (W,H) 를 224×224 로 **직접** 리사이즈하므로(크롭 없음),
GT 박스는 축별 선형 스케일 `x·224/W, y·224/H` 로 변환하면 정확히 정렬됩니다.

In [ ]:
import json
r0 = eval_df.iloc[0]
print(type(r0['gt_boxes']), r0['gt_boxes'])

In [ ]:
def to_xyxy(b, fmt):
    x, y, c, d = float(b[0]), float(b[1]), float(b[2]), float(b[3])
    if fmt == 'xywh':
        return [x, y, x + c, y + d]
    return [x, y, c, d]            # 이미 xyxy

def scale_box(box_xyxy, W, H, size=224):
    sx, sy = size / W, size / H
    x1, y1, x2, y2 = box_xyxy
    return [x1 * sx, y1 * sy, x2 * sx, y2 * sy]

def box_iou(a, b):
    ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

# 좌표계 자동 점검: 임의 1장의 GT 박스가 이미지 경계를 크게 벗어나면 포맷 의심
def sanity_check_format(n=20):
    bad = 0; checked = 0
    for _, r in eval_df.head(n).iterrows():
        W, H = Image.open(r['file_path']).size
        for b in r['gt_boxes']:
            x1, y1, x2, y2 = to_xyxy(b, cfg.bbox_format)
            checked += 1
            if x2 > W * 1.05 or y2 > H * 1.05 or x2 <= x1 or y2 <= y1:
                bad += 1
    print(f'bbox 포맷 점검: {checked}개 중 경계초과/역전 {bad}개', end='  ')
    if checked and bad / checked > 0.2:
        print('⚠️ bbox_format 이 틀렸을 수 있습니다. xywh ↔ xyxy 바꿔보세요.')
    else:
        print('✓ 정상 범위')

sanity_check_format()

## 7. 모델 재구성 & 체크포인트 로드

B0/B1/B2MS 세 아키텍처를 정의하고 `MODEL_VARIANT` 에 맞춰 build 후 `best.pt` 로드. Grad-CAM 타깃은 셋 다 `backbone.layer4`.

In [ ]:
# ── B0 ───────────────────────────────────────────────────────────
class PlainResNet(nn.Module):
    def __init__(self, num_main=3, pretrained=False):
        super().__init__()
        w = ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        self.backbone = resnet50(weights=w)
        in_feat = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.head_main = nn.Linear(in_feat, num_main)
    def forward(self, x):
        return self.head_main(self.backbone(x))

# ── B1 (단일스케일 텍스처) ───────────────────────────────────────
class TextureAwareBlock(nn.Module):
    def __init__(self, mid_ch=32, out_ch=64):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(3, mid_ch, 3, 1, 1, bias=False), nn.BatchNorm2d(mid_ch), nn.ReLU(True),
            nn.Conv2d(mid_ch, out_ch, 3, 2, 1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, 2, 1, bias=False), nn.BatchNorm2d(out_ch))
    def forward(self, x):
        return self.body(x)

# ── B2-MS (멀티스케일 텍스처) ────────────────────────────────────
class MultiScaleTextureBlock(nn.Module):
    def __init__(self, mid_ch=32, out_ch=64):
        super().__init__()
        self.fine   = nn.Conv2d(3, mid_ch, 3, 1, 1, bias=False)
        self.coarse = nn.Conv2d(3, mid_ch, 3, 1, 2, dilation=2, bias=False)
        self.fuse_bn = nn.BatchNorm2d(mid_ch * 2); self.relu = nn.ReLU(True)
        self.down = nn.Sequential(
            nn.Conv2d(mid_ch * 2, out_ch, 3, 2, 1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, 2, 1, bias=False), nn.BatchNorm2d(out_ch))
    def forward(self, x):
        t = torch.cat([self.fine(x), self.coarse(x)], dim=1)
        t = self.relu(self.fuse_bn(t))
        return self.down(t)

class TextureResNet(nn.Module):
    """B1/B2-MS 공용: block_cls 만 바꿔 끼움 (융합 α·백본·헤드 동일)."""
    def __init__(self, num_main=3, block_cls=TextureAwareBlock, texture_mid_ch=32, pretrained=False):
        super().__init__()
        w = ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        self.backbone = resnet50(weights=w)
        in_feat = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.texture_block = block_cls(mid_ch=texture_mid_ch, out_ch=64)
        self.texture_alpha = nn.Parameter(torch.zeros(1))
        self.head_main = nn.Linear(in_feat, num_main)
    def forward(self, x):
        bb = self.backbone
        s = bb.maxpool(bb.relu(bb.bn1(bb.conv1(x))))
        s = s + self.texture_alpha * self.texture_block(x)
        f = bb.layer4(bb.layer3(bb.layer2(bb.layer1(s))))
        f = torch.flatten(bb.avgpool(f), 1)
        return self.head_main(f)

# ── IMPROVED (B0 백본 + Dropout 헤드; Focal/정규화/임계값 학습본) ──
class ImprovedResNet(nn.Module):
    def __init__(self, num_main=3, dropout_p=0.3, pretrained=False):
        super().__init__()
        w = ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        self.backbone = resnet50(weights=w)
        in_feat = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.head_main = nn.Sequential(nn.Dropout(dropout_p), nn.Linear(in_feat, num_main))
    def forward(self, x):
        return self.head_main(self.backbone(x))

def build_model(variant):
    if variant == 'B0':
        return PlainResNet(num_main=len(cfg.main_classes))
    if variant == 'B1':
        return TextureResNet(num_main=len(cfg.main_classes),
                             block_cls=TextureAwareBlock, texture_mid_ch=cfg.texture_mid_ch)
    if variant == 'B2MS':
        return TextureResNet(num_main=len(cfg.main_classes),
                             block_cls=MultiScaleTextureBlock, texture_mid_ch=cfg.texture_mid_ch)
    if variant == 'IMPROVED':
        return ImprovedResNet(num_main=len(cfg.main_classes), dropout_p=cfg.dropout_p)
    if variant == 'B2MS_IMP':
        return MSImprovedResNet(num_main=len(cfg.main_classes),
                                texture_mid_ch=cfg.texture_mid_ch, dropout_p=0.3)
    raise ValueError(variant)
class MSImprovedResNet(nn.Module):
    """멀티스케일 텍스처 + Dropout 헤드 (B2MS + 개선 레시피 학습본)."""
    def __init__(self, num_main=3, texture_mid_ch=32, dropout_p=0.3, pretrained=False):
        super().__init__()
        w = ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        self.backbone = resnet50(weights=w)
        in_feat = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.texture_block = MultiScaleTextureBlock(mid_ch=texture_mid_ch, out_ch=64)
        self.texture_alpha = nn.Parameter(torch.zeros(1))
        self.head_main = nn.Sequential(nn.Dropout(dropout_p), nn.Linear(in_feat, num_main))
    def forward(self, x):
        bb = self.backbone
        s = bb.maxpool(bb.relu(bb.bn1(bb.conv1(x))))
        s = s + self.texture_alpha * self.texture_block(x)
        f = bb.layer4(bb.layer3(bb.layer2(bb.layer1(s))))
        f = torch.flatten(bb.avgpool(f), 1)
        return self.head_main(f)
model = build_model(cfg.MODEL_VARIANT).to(DEVICE)

ckpt = torch.load(CKPT, map_location=DEVICE)
state = ckpt.get('model', ckpt.get('state_dict', ckpt)) if isinstance(ckpt, dict) else ckpt
missing, unexpected = model.load_state_dict(state, strict=False)
model.eval()
print('Loaded:', CKPT)
print('  missing keys   :', len(missing), '| unexpected keys:', len(unexpected))
if len(missing) > 5:
    print('  WARNING: missing 가 많습니다 — MODEL_VARIANT 와 체크포인트가 맞는지 확인하세요.')

TARGET_LAYER = model.backbone.layer4   # 세 변형 공통


## 8. Grad-CAM 구현 (외부 라이브러리 불필요)

`layer4` 출력 활성 \(A\) 와 그에 대한 클래스 점수의 그래디언트를 후크로 받아,
채널 가중치 \(w_k = \mathrm{GAP}(\partial y^c / \partial A^k)\), 가중합 후 ReLU →
224 로 업샘플 → [0,1] 정규화. (Selvaraju et al., 2017)

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.acts = None
        self.grads = None
        target_layer.register_forward_hook(self._fwd)
        target_layer.register_full_backward_hook(self._bwd)
    def _fwd(self, m, i, o):
        self.acts = o.detach()
    def _bwd(self, m, gi, go):
        self.grads = go[0].detach()
    def __call__(self, x, class_idx):
        self.model.zero_grad(set_to_none=True)
        out = self.model(x)
        logits = out[0] if isinstance(out, (tuple, list)) else out  # B2 대응
        if class_idx is None:
            class_idx = int(logits.argmax(1))
        score = logits[0, class_idx]
        score.backward()
        w = self.grads.mean(dim=(2, 3), keepdim=True)        # (1,C,1,1)
        cam = F.relu((w * self.acts).sum(1, keepdim=True))   # (1,1,7,7)
        cam = F.interpolate(cam, size=x.shape[-2:], mode='bilinear', align_corners=False)
        cam = cam[0, 0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.cpu().numpy(), int(class_idx)

def cam_to_box(cam, thr):
    '''CAM -> 임계 후 최대 연결요소의 bounding box (xyxy). 비면 None.'''
    mask = cam >= thr * cam.max()
    lbl, n = ndimage.label(mask)
    if n == 0:
        return None
    sizes = ndimage.sum(mask, lbl, index=range(1, n + 1))
    big = int(np.argmax(sizes)) + 1
    ys, xs = np.where(lbl == big)
    return [float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())]

gradcam = GradCAM(model, TARGET_LAYER)
print('Grad-CAM ready. target =', TARGET_LAYER.__class__.__name__)

## 9. 단일 이미지 데모 — CAM 오버레이 + GT(초록)/예측(빨강) 박스

In [ ]:
def explain_one(row):
    img = Image.open(row['file_path']).convert('RGB')
    W, H = img.size
    x = eval_tf(img).unsqueeze(0).to(DEVICE)
    gt_idx = int(row['main_idx'])
    cls = gt_idx if cfg.cam_class == 'gt' else None
    cam, pred = gradcam(x, class_idx=cls)
    gtb = [scale_box(to_xyxy(b, cfg.bbox_format), W, H) for b in row['gt_boxes']]
    pbox = cam_to_box(cam, cfg.cam_threshold)
    return denormalize(eval_tf(img)), cam, gtb, pbox, gt_idx, pred

row = eval_df.iloc[0]
base, cam, gtb, pbox, gt_idx, pred = explain_one(row)
fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
ax[0].imshow(base); ax[0].set_title('input'); ax[0].axis('off')
ax[1].imshow(base); ax[1].imshow(cam, cmap='jet', alpha=0.45)
for g in gtb:
    ax[1].add_patch(patches.Rectangle((g[0], g[1]), g[2]-g[0], g[3]-g[1],
                    fill=False, edgecolor='lime', lw=2))
if pbox:
    ax[1].add_patch(patches.Rectangle((pbox[0], pbox[1]), pbox[2]-pbox[0], pbox[3]-pbox[1],
                    fill=False, edgecolor='red', lw=2, linestyle='--'))
ax[1].set_title(f'Grad-CAM | GT={cfg.main_classes[gt_idx]} / Pred={cfg.main_classes[pred]}')
ax[1].axis('off'); plt.tight_layout(); plt.show()

## 10. WSOL 정량 평가 — IoU / Loc@0.5 / Pointing-Game

`cam_class='gt'` 면 **GT-Known Localization**(정답 클래스로 CAM 생성 → 위치추정 능력 자체를 측정),
`'pred'` 면 분류까지 맞아야 하는 Top-1 Loc. 기본은 `gt` 입니다.
표본이 많으면 `cfg.max_eval` 로 상한을 둡니다.

In [ ]:
rows_eval = eval_df if cfg.max_eval is None else eval_df.iloc[:cfg.max_eval]
records = []
t0 = time.time()
for k, (_, row) in enumerate(rows_eval.iterrows()):
    img = Image.open(row['file_path']).convert('RGB')
    W, H = img.size
    x = eval_tf(img).unsqueeze(0).to(DEVICE)
    gt_idx = int(row['main_idx'])
    cls = gt_idx if cfg.cam_class == 'gt' else None
    cam, pred = gradcam(x, class_idx=cls)
    gtb = [scale_box(to_xyxy(b, cfg.bbox_format), W, H) for b in row['gt_boxes']]
    pbox = cam_to_box(cam, cfg.cam_threshold)
    iou = max((box_iou(pbox, g) for g in gtb), default=0.0) if pbox else 0.0
    py, px = np.unravel_index(int(np.argmax(cam)), cam.shape)
    hit = any(g[0] <= px <= g[2] and g[1] <= py <= g[3] for g in gtb)
    records.append(dict(main=row['main'], iou=float(iou),
                        hit=bool(hit), correct=bool(pred == gt_idx)))
    if (k + 1) % 100 == 0:
        print(f'  {k+1}/{len(rows_eval)}  ({time.time()-t0:.0f}s)')

res = pd.DataFrame(records)
def summarize(d):
    return dict(n=int(len(d)),
                mean_IoU=round(float(d['iou'].mean()), 4),
                Loc_at_0p5=round(float((d['iou'] >= 0.5).mean()), 4),
                pointing_game=round(float(d['hit'].mean()), 4))

overall = summarize(res)
per_class = {c: summarize(res[res['main'] == c]) for c in cfg.main_classes if (res['main'] == c).any()}
among_correct = summarize(res[res['correct']]) if res['correct'].any() else {}

metrics = dict(variant=cfg.MODEL_VARIANT, cam_class=cfg.cam_class,
               cam_threshold=cfg.cam_threshold, bbox_format=cfg.bbox_format,
               overall=overall, per_class=per_class, among_correct=among_correct)
print(json.dumps(metrics, ensure_ascii=False, indent=2))
with open(OUT / 'xai_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)
res.to_csv(OUT / 'xai_per_sample.csv', index=False)
print('\nSaved →', OUT / 'xai_metrics.json')

## 11. 정성 시각화 그리드 — 잘 짚은 사례 vs 빗나간 사례

IoU 상위/하위 사례를 나란히 보여줘 "모델이 오염·손상 부위를 실제로 보는가"를
정성적으로 해석합니다. 빗나간 사례는 오분류 원인 분석의 단서가 됩니다.

In [ ]:
order = res.sort_values('iou', ascending=False)
top_idx = order.head(cfg.n_viz // 2).index.tolist()
bot_idx = order.tail(cfg.n_viz - len(top_idx)).index.tolist()
pick = top_idx + bot_idx

cols = 4
rows_n = math.ceil(len(pick) / cols)
fig, axes = plt.subplots(rows_n, cols, figsize=(cols * 3.2, rows_n * 3.2))
axes = np.array(axes).reshape(-1)
for ax in axes:
    ax.axis('off')
for ax, ridx in zip(axes, pick):
    row = rows_eval.iloc[ridx]
    base, cam, gtb, pbox, gt_idx, pred = explain_one(row)
    ax.imshow(base); ax.imshow(cam, cmap='jet', alpha=0.45)
    for g in gtb:
        ax.add_patch(patches.Rectangle((g[0], g[1]), g[2]-g[0], g[3]-g[1],
                     fill=False, edgecolor='lime', lw=1.8))
    if pbox:
        ax.add_patch(patches.Rectangle((pbox[0], pbox[1]), pbox[2]-pbox[0], pbox[3]-pbox[1],
                     fill=False, edgecolor='red', lw=1.8, linestyle='--'))
    ok = '✓' if pred == gt_idx else '✗'
    ax.set_title(f"IoU={res.loc[ridx,'iou']:.2f} {ok} {cfg.main_classes[pred]}", fontsize=9)
plt.suptitle(f'{cfg.MODEL_VARIANT}: Grad-CAM vs 오염/손상 GT (초록=GT, 빨강=예측)', y=1.005)
plt.tight_layout()
plt.savefig(OUT / 'gradcam_examples.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved →', OUT / 'gradcam_examples.png')

## 12. 해석 & 평가항목 매핑

**B0/B1/B2 비교법** — 세 변형 각각에 대해 `MODEL_VARIANT` 만 바꿔 이 노트북을 실행하면
`runs/B4_xai/<변형>/xai_metrics.json` 이 생깁니다. 셋을 모아 한 표로:

| 변형 | mean IoU | Loc@0.5 | Pointing | 해석 |
|---|---|---|---|---|
| B0 | … | … | … | 기준 |
| B1 | … | … | … | 질감 보존이 **오염 부위**에 더 집중하게 했나? |
| B2 | … | … | … | 종류 보조과제가 위치추정에 도움/방해? |

> **가설 연결**: B1 의 mean IoU·Pointing 이 B0 보다 높으면 "텍스처 블록이 단순 정확도뿐 아니라
> *모델이 보는 위치*까지 GT(오염/손상)에 가깝게 만들었다"는 **설명가능성 근거**가 됩니다.

**채점 루브릭 충족 포인트**
- **시각화·해석 (15%)**: Grad-CAM 오버레이 + 정성 그리드 → "모델이 어디를 보는가" 직접 제시.
- **성능 분석·최적화 (15%)**: IoU·Loc@0.5·Pointing 은 분류 정확도와 다른 **비전 전용 위치추정 지표**.
- **모델 설계 (40%)**: B1/B2 의 구조 변경이 *위치추정 품질*에까지 영향을 주는지로 설계 타당성 입증.

**주의/한계 (보고서에 명시 권장)**
- GT 박스는 오염/손상 위치이지 "분류 정답 근거"와 100% 일치하진 않음 → Pointing-Game 이 IoU 보다 강건.
- `cam_threshold` 에 IoU 가 민감 → 0.3/0.5/0.7 스윕 결과를 부록에 같이 두면 견고.
- 단일 seed 모델이면 점수에 분산 존재 → 가능하면 seed 평균.
